# Regularized Regression on Phishing Datasets
This notebook processes two datasets:
1. `StealthPhisher2025(2).csv`
2. `PhiUSIIL_Phishing_URL_Dataset(1).csv`

It applies **Linear Regression**, **Lasso Regression (L1)**, **Ridge Regression (L2)**, and **Elastic Net Regression (L1 + L2)** to compare how regularization affects model performance.

In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from pandas.api.types import is_numeric_dtype 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## 1. Defining the Evaluation Function
This function loads the data, imputes missing values, separates and encodes categorical and continuous features, and then fits four different regression models.

In [18]:
def evaluate_regression_models(file_path, target_col, cols_to_drop):
    print(f"\n{'='*60}\nEvaluating Models on {file_path}\n{'='*60}")
    
    # 1. Load and Clean Data
    df = pd.read_csv(file_path)
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
    

# Handle missing values safely
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            # If it's a number, use median
            df[col].fillna(df[col].median(), inplace=True)
        else:
            # If it's text, categorical, or string, use mode
            df[col].fillna(df[col].mode()[0], inplace=True)
            
    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    if y.dtype == 'object' or y.dtype == 'string':
        encoder = LabelEncoder()
        y = encoder.fit_transform(y)
    
    # Identify Categorical and Continuous Features
    categorical_features = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() < 10]
    continuous_features = [col for col in X.columns if col not in categorical_features]
    
    # 2. Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), continuous_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ])
    
    X_processed = preprocessor.fit_transform(X)
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)
    
    # 3. Define Models
    models = {
        "Standard Linear Regression": LinearRegression(),
        "Lasso (L1) Regularization": Lasso(alpha=0.1, random_state=42),
        "Ridge (L2) Regularization": Ridge(alpha=1.0, random_state=42),
        "Elastic Net (L1+L2)": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)
    }
    
    # 4. Fit and Evaluate
    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        results.append({"Model": name, "MSE": mse, "R2 Score": r2})
        
    # Display Results
    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))
    print("\n")

## 2. Analysis of `StealthPhisher2025(2).csv`

In [19]:
stealth_file = 'StealthPhisher2025(2).csv'
cols_to_drop_stealth = ['URL', 'Domain', 'TLD', 'HasTitle', 'Title']
evaluate_regression_models(stealth_file, 'Label', cols_to_drop_stealth)


Evaluating Models on StealthPhisher2025(2).csv
                     Model      MSE  R2 Score
Standard Linear Regression 0.016972  0.931959
 Lasso (L1) Regularization 0.086207  0.654398
 Ridge (L2) Regularization 0.016972  0.931961
       Elastic Net (L1+L2) 0.056041  0.775332




## 3. Analysis of `PhiUSIIL_Phishing_URL_Dataset(1).csv`

In [20]:
phiusiil_file = 'PhiUSIIL_Phishing_URL_Dataset(1).csv'
cols_to_drop_phiusiil = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
evaluate_regression_models(phiusiil_file, 'label', cols_to_drop_phiusiil)


Evaluating Models on PhiUSIIL_Phishing_URL_Dataset(1).csv
                     Model      MSE  R2 Score
Standard Linear Regression 0.016928  0.930802
 Lasso (L1) Regularization 0.072836  0.702260
 Ridge (L2) Regularization 0.016934  0.930779
       Elastic Net (L1+L2) 0.043657  0.821537


